# 02 · So sánh 6 hàm mất mát

Yêu cầu bắt buộc của đề bài. Kiến trúc giữ **cố định** (`transpose` +
`full skip`), chỉ đổi đúng một biến là hàm mất mát.

Sáu lượt chạy, mỗi lượt 20–30 phút trên T4. Nên chia hai tài khoản Colab:
tài khoản 1 chạy `bce`, `weighted_bce`, `dice`; tài khoản 2 chạy `bce_dice`,
`focal`, `tversky`. Cuối cùng gộp hai file `logs/runs.csv` lại.

Người phụ trách: **SV A** chạy, **SV B** tổng hợp.

In [1]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

Thư mục làm việc: /home/quangngo/Documents/TaiLieuThacSi/Hoc_sau_va_ung_dung/btl/files/unet-kvasir
Thiết bị: cpu


In [2]:
base = Config(up_mode="transpose", skip_mode="full", epochs=40, seed=42)

# Đổi lát cắt này khi chia việc giữa hai tài khoản Colab
MY_LOSSES = LOSS_NAMES            # hoặc LOSS_NAMES[:3] / LOSS_NAMES[3:]
configs = [base.replace(loss_name=name, tag="loss_sweep") for name in MY_LOSSES]

for c in configs:
    print(c.run_id)

bce-transpose-full-s42-ef40ae
weighted_bce-transpose-full-s42-c8d926
dice-transpose-full-s42-b12a50
bce_dice-transpose-full-s42-3790b4
focal-transpose-full-s42-c03c02
tversky-transpose-full-s42-f6e06a


In [ ]:
results = run_sweep(configs)


----- [1/6] -----
=== bce-transpose-full-s42-ef40ae === 31,037,633 tham số, thiết bị cpu
epoch   1/40 | train 0.5393 | val 0.3864 | val dice 0.0311 | val iou 0.0166 *


## Bảng so sánh

Đọc thẳng từ `logs/runs.csv` chứ không gõ số bằng tay — đây là cách duy nhất
đảm bảo báo cáo không lệch với log.

In [ ]:
import pandas as pd

df = RunLogger(base.log_csv).to_dataframe()
df = df[df["tag"] == "loss_sweep"]
cols = ["loss_name", "test_dice", "test_iou", "test_precision", "test_recall",
        "best_epoch", "train_time_min"]
table = df[cols].astype({c: float for c in cols[1:]}).sort_values("test_dice",
                                                                  ascending=False)
table.round(4)

In [ ]:
print(table.round(4).to_markdown(index=False))   # dán thẳng vào báo cáo

## Đường cong val Dice của các hàm mất mát

In [ ]:
from src.viz import plot_loss_curves

histories = {r["loss_name"]: r["history"] for r in results.values()
             if not r.get("skipped")}
fig = plot_loss_curves(histories, key="val_dice",
                       save_path=f"{base.fig_dir}/loss_sweep_valdice.png")

## Câu hỏi cần trả lời trong báo cáo

1. Dice-based loss có thắng BCE không, và chênh bao nhiêu? Chênh đó có lớn
   hơn dao động giữa các seed không?
2. Weighted BCE và Focal đều nhắm vào mất cân bằng lớp — cái nào hiệu quả hơn
   ở đây, và tại sao?
3. Tversky với `beta > alpha` có làm recall tăng và precision giảm đúng như
   kỳ vọng lý thuyết không? Kiểm bằng đúng hai cột đó trong bảng.

## Lặp seed cho 3 cấu hình tốt nhất

Không bắt buộc với đề tài 5, nhưng đây là chỗ ăn điểm mục *Thí nghiệm và
phân tích*: nó cho phép nói câu "khác biệt này nằm trong nhiễu".

In [ ]:
top3 = table["loss_name"].head(3).tolist()
seed_configs = [base.replace(loss_name=name, seed=1337, tag="loss_seed2")
                for name in top3]
# results_seed2 = run_sweep(seed_configs)